# 01. 전처리 — 원본 데이터를 학습 가능한 표(wide table)로 만들기

**이 노트북이 하는 일 (Why)**
- 지금 GFS/LDAPS 예보 데이터는 예측 대상 시각(`forecast_kst_dtm`) 하나마다 격자(`grid_id`)별로 여러 행이 이어 붙은 **long 구조**예요. long 구조는 "한 시각에 여러 줄"이라 모델에 바로 넣을 수 없어요 (모델은 "한 시각 = 한 행"을 기대합니다).
- 이 노트북에서는 long 구조를 **wide 구조**(한 시각 = 한 행, 격자별 값은 옆으로 나열)로 바꾸고, 실제 발전량(`train_labels.csv`) 또는 제출 골격(`sample_submission.csv`)과 병합해서, 이후 EDA·피처 엔지니어링이 바로 쓸 수 있는 기본 캐시(parquet 파일)를 만듭니다.
- **여기서는 하지 않는 것**: 결측치 채우기, 이상치 제거, 스케일링. 이런 건 다음 단계(EDA로 근거를 모은 뒤)에서 합니다. 이 노트북은 순수하게 "구조를 바꾸고 병합하는" 단계예요.

**입력**
- `data/train/gfs_train.csv`, `data/train/ldaps_train.csv`, `data/train/train_labels.csv`
- `data/test/gfs_test.csv`, `data/test/ldaps_test.csv`, `data/sample_submission.csv`

**출력** (`data/processed/`)
- `train_base_wide.parquet` / `test_base_wide.parquet`: 격자별 컬럼을 그대로 보존한 버전
- `train_base_agg.parquet` / `test_base_agg.parquet`: 격자를 평균·표준편차로 집계한 버전 (스킬 규칙 — 어느 쪽이 나은지는 나중에 모델 비교로 정하므로 둘 다 남긴다)
- `gfs_grid_meta.parquet` / `ldaps_grid_meta.parquet`: 격자별 위도·경도 (나중에 지도 시각화·최근접 격자 판단용)

**train/test 동일 로직**: 이 노트북의 모든 변환은 함수로 만들어서 train과 test에 똑같이 적용합니다 (전처리 스킬 규칙 — 다르게 처리하면 나중에 추론 단계에서 사고가 납니다).

## 0. 셋업
아래 셀은 필요한 패키지를 불러오고, 경로 상수와 시드를 고정합니다. `sys.executable`을 출력해서 지금 커널이 이 프로젝트의 가상환경(venv)인지 확인하세요 — 경로에 `wind_forecast\venv`가 포함되어 있어야 합니다. 커널 선택은 Jupyter 우측 상단에서 `wind_forecast (venv)`로 맞춰주세요.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

# 이 노트북(notebooks/01_preprocessing.ipynb)이 어디서 실행되든 REPO_ROOT를 찾아낸다
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

DATA_DIR = REPO_ROOT / "data"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)

python executable: c:\Users\cho03\Desktop\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: c:\Users\cho03\Desktop\wind_forecast


## 0-1. 검증 분리 기준일 (상수로 고정)
`HANDOFF.md`에 기록된 2단계 결정에 따라 학습/검증 분리 날짜를 여기서 상수로 못박습니다. 앞으로 모든 노트북(EDA, 피처, 모델링)은 이 상수를 그대로 가져다 씁니다 — 노트북마다 다른 날짜를 쓰면 비교가 무의미해지기 때문이에요.

이 노트북 자체는 train 라벨 전체 기간(2022~2024)을 캐시로 만들고, 실제 학습/검증 행 분리는 모델링 단계에서 이 상수로 마스킹해서 적용합니다.

In [2]:
# 학습/검증 분리 기준 (HANDOFF.md 2단계 결정 사항과 동일하게 유지할 것)
TRAIN_START = pd.Timestamp("2022-01-01 01:00:00")
TRAIN_END = pd.Timestamp("2024-01-01 00:00:00")      # 포함
VALID_START = pd.Timestamp("2024-01-01 01:00:00")
VALID_END = pd.Timestamp("2025-01-01 00:00:00")      # 포함, train_labels.csv 마지막 시각과 동일

print(f"train: {TRAIN_START} ~ {TRAIN_END}")
print(f"valid: {VALID_START} ~ {VALID_END}")

train: 2022-01-01 01:00:00 ~ 2024-01-01 00:00:00
valid: 2024-01-01 01:00:00 ~ 2025-01-01 00:00:00


## 1. 원본 CSV 로드
데이터 설명서에 따르면 모든 CSV는 `UTF-8 with BOM`(`utf-8-sig`)으로 저장돼 있어요. 이 인코딩을 안 맞추면 첫 번째 컬럼명 앞에 이상한 문자(`\ufeff`)가 붙어서 이후 코드가 깨집니다. 시각 컬럼은 로드 직후 바로 `datetime` 타입으로 바꿔서, 이후 모든 병합의 키가 같은 타입이 되게 합니다.

In [3]:
def load_weather_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    df["data_available_kst_dtm"] = pd.to_datetime(df["data_available_kst_dtm"])
    return df


gfs_train_raw = load_weather_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_train_raw = load_weather_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_test_raw = load_weather_csv(TEST_DIR / "gfs_test.csv")
ldaps_test_raw = load_weather_csv(TEST_DIR / "ldaps_test.csv")

labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
labels["kst_dtm"] = pd.to_datetime(labels["kst_dtm"])

sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig", dtype={"forecast_id": str})
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

In [4]:
for name, df in [
    ("gfs_train", gfs_train_raw), ("ldaps_train", ldaps_train_raw),
    ("gfs_test", gfs_test_raw), ("ldaps_test", ldaps_test_raw),
    ("labels", labels), ("sample_submission", sample_submission),
]:
    print(f"{name}: shape={df.shape}")

display(gfs_train_raw.head(5))
display(ldaps_train_raw.head(5))

gfs_train: shape=(236736, 40)
ldaps_train: shape=(420864, 35)
gfs_test: shape=(78840, 40)
ldaps_test: shape=(140160, 35)
labels: shape=(26304, 4)
sample_submission: shape=(8760, 5)


,forecast_kst_dtm,data_available_kst_dtm,grid_id,latitude,longitude,heightAboveGround_10_10u,heightAboveGround_10_10v,heightAboveGround_80_u,heightAboveGround_80_v,heightAboveGround_100_100u,heightAboveGround_100_100v,heightAboveGround_2_2t,heightAboveGround_2_2d,heightAboveGround_2_2r,heightAboveGround_2_2sh,planetaryBoundaryLayer_0_u,planetaryBoundaryLayer_0_v,planetaryBoundaryLayer_0_VRATE,surface_0_dswrf,surface_0_dlwrf,surface_0_prate,surface_0_tp,surface_0_sp,meanSea_0_prmsl,surface_0_gust,lowCloudLayer_0_lcc,middleCloudLayer_0_mcc,highCloudLayer_0_hcc,atmosphere_0_tcc,isobaricInhPa_850_t,isobaricInhPa_850_u,isobaricInhPa_850_v,isobaricInhPa_850_r,isobaricInhPa_700_t,isobaricInhPa_700_u,isobaricInhPa_700_v,isobaricInhPa_500_gh,isobaricInhPa_500_t,isobaricInhPa_500_u,isobaricInhPa_500_v
0,2022-01-01 01:00:00,2021-12-31 13:00:00,1,37.50,128.75,1.817466,1.040068,2.437944,1.258201,2.539397,1.267655,260.3425,249.06181,41.5,0.000583,1.968707,1.032028,0.0,0.0,162.56856,0.0,0.0,93651.410,103163.5,2.601783,0.0,0.0,4.8,5.0,265.62122,5.928218,-3.632051,8.5,260.32092,11.584992,-18.589058,5474.115,243.86014,22.805730,-15.617361
1,2022-01-01 01:00:00,2021-12-31 13:00:00,2,37.50,129.00,2.947466,1.340068,3.747944,1.728201,3.839397,1.797655,262.5225,250.18182,37.7,0.000621,3.168707,1.432028,0.0,0.0,168.66855,0.0,0.0,97209.805,103011.1,4.101783,0.0,0.0,4.6,4.6,265.56122,4.158218,-2.582051,8.3,259.80090,11.744992,-18.769058,5470.083,244.00014,23.905731,-16.017360
2,2022-01-01 01:00:00,2021-12-31 13:00:00,3,37.50,129.25,2.817466,0.800068,2.577944,0.808201,2.509397,0.777655,270.8025,257.20180,34.5,0.001077,2.268707,0.732028,1000.0,0.0,194.16855,0.0,0.0,102629.010,102848.3,2.001783,0.0,0.0,5.0,2.8,264.90125,4.228218,-2.532051,9.5,259.58093,12.354993,-18.999058,5465.347,244.09013,24.905731,-16.317362
3,2022-01-01 01:00:00,2021-12-31 13:00:00,4,37.25,128.75,1.227466,0.190068,2.037944,0.038201,2.159397,-0.002345,259.9825,249.56181,44.9,0.000609,1.368707,0.132028,0.0,0.0,162.26855,0.0,0.0,93747.410,103218.7,1.601783,0.0,0.0,3.7,79.6,265.67123,5.308218,-4.312051,8.2,260.77090,11.854993,-18.309057,5480.051,244.00014,23.605732,-14.717361
4,2022-01-01 01:00:00,2021-12-31 13:00:00,5,37.25,129.00,2.457466,0.540068,3.107944,0.568201,3.129397,0.547655,261.2625,247.91182,34.5,0.000527,2.568707,0.532028,0.0,0.0,163.16855,0.0,0.0,93449.805,103061.9,2.801783,0.0,0.0,3.7,60.2,266.12122,3.528218,-3.132051,7.8,260.33093,11.714993,-18.689058,5476.259,244.14014,24.205730,-15.317362


,forecast_kst_dtm,data_available_kst_dtm,grid_id,latitude,longitude,heightAboveGround_10_10u,heightAboveGround_10_10v,heightAboveGround_50_50MUmax,heightAboveGround_50_50MUmin,heightAboveGround_50_50MVmax,heightAboveGround_50_50MVmin,heightAboveGround_5_XBLWS,heightAboveGround_5_YBLWS,heightAboveGround_2_t,heightAboveGround_2_dpt,heightAboveGround_2_r,heightAboveGround_2_q,surface_0_sp,meanSea_0_prmsl,etc_0_blh,surface_0_NDNSW,surface_0_NDNLW,heightAboveGround_2_SWDIR,heightAboveGround_2_SWDIF,etc_0_hcc,etc_0_mcc,etc_0_lcc,etc_0_VLCDC,surface_0_avg_lsprate,surface_0_lssrate,surface_0_ncpcp,surface_0_snol,surface_0_SNOM,surface_0_lsm,surface_0_h
0,2022-01-01 01:00:00,2021-12-31 13:00:00,1,37.3032,128.9443,5.411171,1.591679,8.162608,7.618145,2.164301,1.766520,0.013109,0.191541,260.74554,246.23834,32.642956,0.000488,90628.91,102664.920,177.19792,0.0,-95.399080,0.0,0.0,0.093750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,992.6250
1,2022-01-01 01:00:00,2021-12-31 13:00:00,2,37.3027,128.9617,5.593300,0.727909,8.908702,8.205059,1.596918,1.039957,0.013964,0.130384,261.73870,245.81940,28.693737,0.000488,91273.41,102658.484,177.13542,0.0,-97.274080,0.0,0.0,0.093750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,936.5625
2,2022-01-01 01:00:00,2021-12-31 13:00:00,3,37.3022,128.9790,4.383828,-0.924435,7.793468,7.167461,0.100824,-0.688559,0.014452,-0.041247,262.34515,245.96979,27.598034,0.000488,92078.41,102664.016,162.07292,0.0,-98.266266,0.0,0.0,0.093750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,868.8125
3,2022-01-01 01:00:00,2021-12-31 13:00:00,4,37.2899,128.9263,3.022011,0.641972,5.043956,4.645001,0.879145,0.647379,0.010180,0.032606,260.13812,247.85748,39.834362,0.000488,91446.91,102678.390,144.32292,0.0,-92.856110,0.0,0.0,0.105362,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,926.5000
4,2022-01-01 01:00:00,2021-12-31 13:00:00,5,37.2894,128.9437,4.837441,0.774784,7.277843,6.799786,1.086176,0.836832,0.011400,0.084241,260.00922,246.97370,37.137096,0.000488,90582.91,102672.266,165.01042,0.0,-93.770170,0.0,0.0,0.105606,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,997.6250


## 2. 기간·행 수가 명세서와 맞는지 확인
`data_description.md`에 적힌 기간과 행 수를 그대로 코드로 재확인합니다. 여기서 하나라도 어긋나면, 다음 단계로 넘어가지 않고 원인부터 찾아야 해요 (조용히 넘어가면 나중에 훨씬 찾기 어려운 버그가 됩니다).

In [5]:
assert gfs_train_raw.shape[0] == 236736
assert ldaps_train_raw.shape[0] == 420864
assert labels.shape[0] == 26304
assert gfs_test_raw.shape[0] == 78840
assert ldaps_test_raw.shape[0] == 140160
assert sample_submission.shape[0] == 8760

assert labels["kst_dtm"].min() == pd.Timestamp("2022-01-01 01:00:00")
assert labels["kst_dtm"].max() == pd.Timestamp("2025-01-01 00:00:00")
assert sample_submission["forecast_kst_dtm"].min() == pd.Timestamp("2025-01-01 01:00:00")
assert sample_submission["forecast_kst_dtm"].max() == pd.Timestamp("2026-01-01 00:00:00")

print("모든 행 수·기간이 명세서와 일치합니다.")

모든 행 수·기간이 명세서와 일치합니다.


## 3. 격자 메타데이터(위도·경도) 따로 저장
각 `grid_id`의 위도·경도는 시간이 지나도 바뀌지 않는 고정값이에요 (격자는 물리적 위치니까요). 이 값을 매 시각 행마다 반복해서 들고 있을 필요는 없으니, 격자 하나당 한 행짜리 메타 표로 따로 뽑아둡니다. 나중에 `02_eda.ipynb`에서 지도 위에 격자 위치와 터빈 위치(`info.xlsx`)를 겹쳐 그릴 때 씁니다.

In [6]:
def extract_grid_meta(df):
    meta = (
        df[["grid_id", "latitude", "longitude"]]
        .drop_duplicates()
        .sort_values("grid_id")
        .reset_index(drop=True)
    )
    assert meta["grid_id"].is_unique, "grid_id별 위도/경도가 시간에 따라 바뀝니다 — 가정이 깨졌습니다."
    return meta


gfs_grid_meta = extract_grid_meta(gfs_train_raw)
ldaps_grid_meta = extract_grid_meta(ldaps_train_raw)

print("GFS 격자 수:", len(gfs_grid_meta))
print("LDAPS 격자 수:", len(ldaps_grid_meta))

GFS 격자 수: 9
LDAPS 격자 수: 16


In [7]:
display(gfs_grid_meta)
display(ldaps_grid_meta.head())

,grid_id,latitude,longitude
0,1,37.50,128.75
1,2,37.50,129.00
2,3,37.50,129.25
3,4,37.25,128.75
4,5,37.25,129.00
5,6,37.25,129.25
6,7,37.00,128.75
7,8,37.00,129.00
8,9,37.00,129.25


,grid_id,latitude,longitude
0,1,37.3032,128.9443
1,2,37.3027,128.9617
2,3,37.3022,128.9790
3,4,37.2899,128.9263
4,5,37.2894,128.9437


## 4. train과 test의 격자 구성이 같은지 확인
격자 배치가 train/test에서 달라지면 wide 변환 후 컬럼 이름이 서로 어긋나서, 나중에 학습한 모델을 test에 그대로 못 씁니다. 미리 확인합니다.

In [8]:
gfs_grid_meta_test = extract_grid_meta(gfs_test_raw)
ldaps_grid_meta_test = extract_grid_meta(ldaps_test_raw)

assert gfs_grid_meta.equals(gfs_grid_meta_test), "GFS 격자 구성이 train/test 간 다릅니다."
assert ldaps_grid_meta.equals(ldaps_grid_meta_test), "LDAPS 격자 구성이 train/test 간 다릅니다."

print("train/test 격자 구성이 동일함을 확인했습니다.")

train/test 격자 구성이 동일함을 확인했습니다.


## 5. long → wide 변환 ①: 격자별 컬럼을 그대로 펼치기
지금 구조는 "한 시각에 격자 수만큼 행"이 있는 long 구조예요 (10분 단위 가계부처럼 잘게 쪼개진 것과 비슷하게, 여기선 "시각"이 아니라 "격자"로 잘게 쪼개져 있는 셈입니다). `pivot`을 이용해서 "한 시각 = 한 행"이 되도록 바꾸고, 격자별 값은 `gfs_g1_변수명`처럼 옆으로 나열합니다. `data_available_kst_dtm`은 격자와 상관없이 시각마다 하나뿐이므로 별도로 붙입니다.

In [9]:
def pivot_weather_wide(df, source):
    """long(시각 x 격자) 구조를 wide(시각 하나 = 한 행) 구조로 바꾼다.

    컬럼명 규칙: f"{source}_g{grid_id}_{variable}"
    """
    value_cols = [
        c for c in df.columns
        if c not in ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]
    ]

    wide = df.pivot(index="forecast_kst_dtm", columns="grid_id", values=value_cols)
    wide.columns = [f"{source}_g{grid_id}_{var}" for var, grid_id in wide.columns]
    wide = wide.reset_index()

    avail = df[["forecast_kst_dtm", "data_available_kst_dtm"]].drop_duplicates()
    assert avail["forecast_kst_dtm"].is_unique, "forecast_kst_dtm 하나에 data_available_kst_dtm이 둘 이상입니다."
    avail = avail.rename(columns={"data_available_kst_dtm": f"{source}_data_available_kst_dtm"})

    wide = wide.merge(avail, on="forecast_kst_dtm", how="left")
    return wide

In [10]:
gfs_train_wide = pivot_weather_wide(gfs_train_raw, "gfs")
ldaps_train_wide = pivot_weather_wide(ldaps_train_raw, "ldaps")
gfs_test_wide = pivot_weather_wide(gfs_test_raw, "gfs")
ldaps_test_wide = pivot_weather_wide(ldaps_test_raw, "ldaps")

In [11]:
print("gfs_train_wide:", gfs_train_wide.shape)
print("ldaps_train_wide:", ldaps_train_wide.shape)
print("gfs_test_wide:", gfs_test_wide.shape)
print("ldaps_test_wide:", ldaps_test_wide.shape)
display(gfs_train_wide.head(3))

gfs_train_wide: (26304, 317)
ldaps_train_wide: (26304, 482)
gfs_test_wide: (8760, 317)
ldaps_test_wide: (8760, 482)


,forecast_kst_dtm,gfs_g1_heightAboveGround_10_10u,gfs_g2_heightAboveGround_10_10u,gfs_g3_heightAboveGround_10_10u,gfs_g4_heightAboveGround_10_10u,gfs_g5_heightAboveGround_10_10u,gfs_g6_heightAboveGround_10_10u,gfs_g7_heightAboveGround_10_10u,gfs_g8_heightAboveGround_10_10u,gfs_g9_heightAboveGround_10_10u,gfs_g1_heightAboveGround_10_10v,gfs_g2_heightAboveGround_10_10v,gfs_g3_heightAboveGround_10_10v,gfs_g4_heightAboveGround_10_10v,gfs_g5_heightAboveGround_10_10v,gfs_g6_heightAboveGround_10_10v,gfs_g7_heightAboveGround_10_10v,gfs_g8_heightAboveGround_10_10v,gfs_g9_heightAboveGround_10_10v,gfs_g1_heightAboveGround_80_u,gfs_g2_heightAboveGround_80_u,gfs_g3_heightAboveGround_80_u,gfs_g4_heightAboveGround_80_u,gfs_g5_heightAboveGround_80_u,gfs_g6_heightAboveGround_80_u,...,gfs_g4_isobaricInhPa_500_t,gfs_g5_isobaricInhPa_500_t,gfs_g6_isobaricInhPa_500_t,gfs_g7_isobaricInhPa_500_t,gfs_g8_isobaricInhPa_500_t,gfs_g9_isobaricInhPa_500_t,gfs_g1_isobaricInhPa_500_u,gfs_g2_isobaricInhPa_500_u,gfs_g3_isobaricInhPa_500_u,gfs_g4_isobaricInhPa_500_u,gfs_g5_isobaricInhPa_500_u,gfs_g6_isobaricInhPa_500_u,gfs_g7_isobaricInhPa_500_u,gfs_g8_isobaricInhPa_500_u,gfs_g9_isobaricInhPa_500_u,gfs_g1_isobaricInhPa_500_v,gfs_g2_isobaricInhPa_500_v,gfs_g3_isobaricInhPa_500_v,gfs_g4_isobaricInhPa_500_v,gfs_g5_isobaricInhPa_500_v,gfs_g6_isobaricInhPa_500_v,gfs_g7_isobaricInhPa_500_v,gfs_g8_isobaricInhPa_500_v,gfs_g9_isobaricInhPa_500_v,gfs_data_available_kst_dtm
0,2022-01-01 01:00:00,1.817466,2.947466,2.817466,1.227466,2.457466,2.247466,0.997466,1.617466,2.827466,1.040068,1.340068,0.800068,0.190068,0.540068,0.490068,-1.499932,-0.099932,0.680068,2.437944,3.747944,2.577944,2.037944,3.107944,2.347944,...,244.00014,244.14014,244.28014,244.71014,244.25014,244.36014,22.805730,23.905731,24.905731,23.605732,24.205730,25.005732,25.305730,24.905731,25.005732,-15.617361,-16.017360,-16.317362,-14.717361,-15.317362,-15.517362,-14.117361,-13.717361,-13.717361,2021-12-31 13:00:00
1,2022-01-01 02:00:00,1.781655,2.861655,2.851655,1.401655,2.391655,2.091655,1.011655,1.651655,2.831655,1.069929,1.289929,0.819929,0.319929,0.629929,0.459929,-1.390071,0.229929,0.709929,2.365549,3.635549,2.575549,2.225549,2.985549,2.105549,...,244.01562,244.10562,244.12563,244.96562,244.67563,244.67563,21.762934,22.762934,23.362934,22.962933,23.262934,23.862934,25.562933,25.062933,25.062933,-14.392309,-14.892309,-15.292310,-13.392309,-14.092310,-14.392309,-13.492310,-12.892309,-13.192309,2021-12-31 13:00:00
2,2022-01-01 03:00:00,1.869741,2.819741,2.639741,1.359741,2.389741,2.119741,0.679741,1.579741,2.889741,1.178535,1.378535,1.058535,0.308535,0.758535,0.638535,-1.761465,-0.161465,0.578535,2.494646,3.574646,2.344646,2.164646,2.984646,2.134646,...,244.19461,244.40460,244.35461,244.87460,244.55461,244.66461,21.128216,21.628216,21.928217,23.328217,23.228216,23.428217,25.328217,24.628216,24.528217,-12.771304,-13.571304,-14.271304,-12.071304,-12.771304,-13.371305,-13.171305,-12.371305,-12.571304,2021-12-31 13:00:00


## 6. long → wide 변환 ②: 격자를 평균·표준편차로 집계하기
격자별 컬럼을 다 펼치면 변수 개수가 많아져요. 원본 CSV 전체 컬럼 수는 LDAPS 35개, GFS 40개인데, 여기엔 `forecast_kst_dtm`/`data_available_kst_dtm`/`grid_id`/`latitude`/`longitude` 같은 식별자 컬럼 5개가 포함돼 있어요. 이 5개를 뺀 **실제 기상 변수 수**는 LDAPS 30개, GFS 35개이고, 이게 격자별로 펼쳐지는 대상입니다 (LDAPS: 30개 × 격자 16개 = 480개 + 시각/발표시각 2개 = 482개, GFS: 35개 × 격자 9개 = 315개 + 2개 = 317개). 대안으로, 같은 시각의 격자들을 평균·표준편차로 하나로 뭉치는 방법도 있습니다. 표준편차는 "격자 간 차이가 큰 시각(=지역별로 바람이 다르게 부는 시각)"을 알려주는 정보라서 그 자체로 의미가 있어요. 어느 버전(격자별 vs 집계)이 모델 성능에 더 좋은지는 나중에(모델 선택 단계) 직접 비교해서 정합니다 — 지금은 둘 다 만들어 둡니다.

In [12]:
def aggregate_weather(df, source):
    """격자 간 평균·표준편차로 집계한다. 컬럼명 규칙: f"{source}_mean_{var}", f"{source}_std_{var}"."""
    value_cols = [
        c for c in df.columns
        if c not in ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]
    ]

    agg = df.groupby("forecast_kst_dtm")[value_cols].agg(["mean", "std"])
    agg.columns = [f"{source}_{stat}_{var}" for var, stat in agg.columns]
    agg = agg.reset_index()

    avail = df[["forecast_kst_dtm", "data_available_kst_dtm"]].drop_duplicates()
    avail = avail.rename(columns={"data_available_kst_dtm": f"{source}_data_available_kst_dtm"})
    agg = agg.merge(avail, on="forecast_kst_dtm", how="left")
    return agg

In [13]:
gfs_train_agg = aggregate_weather(gfs_train_raw, "gfs")
ldaps_train_agg = aggregate_weather(ldaps_train_raw, "ldaps")
gfs_test_agg = aggregate_weather(gfs_test_raw, "gfs")
ldaps_test_agg = aggregate_weather(ldaps_test_raw, "ldaps")

In [14]:
print("gfs_train_agg:", gfs_train_agg.shape)
print("ldaps_train_agg:", ldaps_train_agg.shape)
display(ldaps_train_agg.head(3))

gfs_train_agg: (26304, 72)
ldaps_train_agg: (26304, 62)


,forecast_kst_dtm,ldaps_mean_heightAboveGround_10_10u,ldaps_std_heightAboveGround_10_10u,ldaps_mean_heightAboveGround_10_10v,ldaps_std_heightAboveGround_10_10v,ldaps_mean_heightAboveGround_50_50MUmax,ldaps_std_heightAboveGround_50_50MUmax,ldaps_mean_heightAboveGround_50_50MUmin,ldaps_std_heightAboveGround_50_50MUmin,ldaps_mean_heightAboveGround_50_50MVmax,ldaps_std_heightAboveGround_50_50MVmax,ldaps_mean_heightAboveGround_50_50MVmin,ldaps_std_heightAboveGround_50_50MVmin,ldaps_mean_heightAboveGround_5_XBLWS,ldaps_std_heightAboveGround_5_XBLWS,ldaps_mean_heightAboveGround_5_YBLWS,ldaps_std_heightAboveGround_5_YBLWS,ldaps_mean_heightAboveGround_2_t,ldaps_std_heightAboveGround_2_t,ldaps_mean_heightAboveGround_2_dpt,ldaps_std_heightAboveGround_2_dpt,ldaps_mean_heightAboveGround_2_r,ldaps_std_heightAboveGround_2_r,ldaps_mean_heightAboveGround_2_q,ldaps_std_heightAboveGround_2_q,...,ldaps_mean_heightAboveGround_2_SWDIF,ldaps_std_heightAboveGround_2_SWDIF,ldaps_mean_etc_0_hcc,ldaps_std_etc_0_hcc,ldaps_mean_etc_0_mcc,ldaps_std_etc_0_mcc,ldaps_mean_etc_0_lcc,ldaps_std_etc_0_lcc,ldaps_mean_etc_0_VLCDC,ldaps_std_etc_0_VLCDC,ldaps_mean_surface_0_avg_lsprate,ldaps_std_surface_0_avg_lsprate,ldaps_mean_surface_0_lssrate,ldaps_std_surface_0_lssrate,ldaps_mean_surface_0_ncpcp,ldaps_std_surface_0_ncpcp,ldaps_mean_surface_0_snol,ldaps_std_surface_0_snol,ldaps_mean_surface_0_SNOM,ldaps_std_surface_0_SNOM,ldaps_mean_surface_0_lsm,ldaps_std_surface_0_lsm,ldaps_mean_surface_0_h,ldaps_std_surface_0_h,ldaps_data_available_kst_dtm
0,2022-01-01 01:00:00,5.297585,1.122211,-0.173397,0.846778,8.421367,1.778850,7.729351,1.581838,0.158442,1.010439,-0.198080,0.985325,0.011530,0.002070,-0.025156,0.113568,261.071289,0.868283,246.420777,0.738217,32.464123,4.436594,0.000488,0.000000,...,0.0,0.0,0.102504,0.006823,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,943.527344,44.790891,2021-12-31 13:00:00
1,2022-01-01 02:00:00,4.346036,1.134539,-0.407328,0.959052,8.261779,1.800144,6.641635,1.736260,0.045467,0.994708,-0.761471,0.954384,0.011588,0.000445,-0.044585,0.101151,261.361694,0.905585,246.386843,1.246330,31.972141,6.587127,0.000504,0.000050,...,0.0,0.0,0.152085,0.005886,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,943.527344,44.790891,2021-12-31 13:00:00
2,2022-01-01 03:00:00,4.191131,1.046957,-0.210821,0.966784,6.637347,1.719632,5.875278,1.635564,-0.297825,1.075239,-0.642738,1.053933,0.008403,0.000910,-0.043406,0.087139,261.696288,0.877159,245.935806,1.252609,29.795175,6.282153,0.000496,0.000025,...,0.0,0.0,0.160141,0.005705,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,943.527344,44.790891,2021-12-31 13:00:00


## 7. 기준 시간축(라벨/제출 골격)에 기상 데이터 병합
병합 순서가 중요해요: **라벨(train) 또는 제출 골격(test)을 기준 축**으로 두고, 기상 데이터를 왼쪽 조인(left join)으로 붙입니다. 반대로 하면(기상 데이터를 기준으로 라벨을 붙이면) 혹시 기상 쪽에 빠진 시각이 있을 때 라벨 행이 조용히 사라질 수 있어요.

**참고: train과 test의 최종 컬럼 수는 2개 차이가 나는 게 정상입니다.** 기준 축 자체가 다르기 때문이에요.
- train 기준 축(`labels`): `forecast_kst_dtm` + `kpx_group_1/2/3` = 4개 컬럼
- test 기준 축(`sample_submission`): `forecast_id` + `forecast_kst_dtm` = 2개 컬럼

기상 데이터(GFS/LDAPS wide 또는 agg) 컬럼 수는 train/test가 완전히 동일하므로, 최종 표의 컬럼 수 차이 = 기준 축 컬럼 수 차이 = 4 − 2 = 2개(라벨 3개 − `forecast_id` 1개)입니다. 즉 "기상 변수 개수가 다른 것"이 아니라 "라벨/`forecast_id`처럼 애초에 한쪽에만 있는 컬럼" 때문에 생기는 차이예요. 이 사실은 10번 셀에서 라벨 컬럼과 `forecast_id`를 미리 제외하고 기상 컬럼만 비교하는 `assert`로 확인합니다.

In [15]:
def build_base_table(base_df, base_time_col, gfs_wide, ldaps_wide, gfs_agg, ldaps_agg):
    df = base_df.rename(columns={base_time_col: "forecast_kst_dtm"}).copy()

    wide = df.merge(gfs_wide, on="forecast_kst_dtm", how="left")
    wide = wide.merge(ldaps_wide, on="forecast_kst_dtm", how="left")

    agg = df.merge(gfs_agg, on="forecast_kst_dtm", how="left")
    agg = agg.merge(ldaps_agg, on="forecast_kst_dtm", how="left")

    return wide, agg

In [16]:
train_base_wide, train_base_agg = build_base_table(
    labels, "kst_dtm", gfs_train_wide, ldaps_train_wide, gfs_train_agg, ldaps_train_agg
)
test_base_wide, test_base_agg = build_base_table(
    sample_submission[["forecast_id", "forecast_kst_dtm"]], "forecast_kst_dtm",
    gfs_test_wide, ldaps_test_wide, gfs_test_agg, ldaps_test_agg
)

In [17]:
print("train_base_wide:", train_base_wide.shape)
print("train_base_agg:", train_base_agg.shape)
print("test_base_wide:", test_base_wide.shape)
print("test_base_agg:", test_base_agg.shape)

train_base_wide: (26304, 801)
train_base_agg: (26304, 136)
test_base_wide: (8760, 799)
test_base_agg: (8760, 134)


## 8. 병합 후 결측치 확인
왼쪽 조인을 했으니, 혹시 기상 데이터 쪽에 빠진 시각이 있으면 그 행의 기상 컬럼이 통째로 결측(NaN)이 됩니다. 여기서는 결측을 채우지 않고 **얼마나 있는지 확인만** 합니다 (채우는 방법은 결측이 어디에 몰려있는지 EDA로 확인한 뒤 근거를 가지고 정합니다).

In [18]:
def report_missing(df, name):
    na_total = int(df.isna().sum().sum())
    na_cols = df.isna().sum()
    na_cols = na_cols[na_cols > 0].sort_values(ascending=False)
    print(f"[{name}] 총 결측 셀 수: {na_total}")
    if len(na_cols) > 0:
        print(na_cols.head(10))
    return na_cols


_ = report_missing(train_base_wide, "train_base_wide")
_ = report_missing(test_base_wide, "test_base_wide")

[train_base_wide] 총 결측 셀 수: 8973
kpx_group_3    8766
kpx_group_1     104
kpx_group_2     103
dtype: int64
[test_base_wide] 총 결측 셀 수: 752
ldaps_g16_surface_0_lsm    3
ldaps_g15_surface_0_lsm    3
ldaps_g14_surface_0_lsm    3
ldaps_g13_surface_0_lsm    3
ldaps_g12_surface_0_lsm    3
ldaps_g11_surface_0_lsm    3
ldaps_g10_surface_0_lsm    3
ldaps_g9_surface_0_lsm     3
ldaps_g8_surface_0_lsm     3
ldaps_g7_surface_0_lsm     3
dtype: int64


## 9. 물리적으로 말이 되는 값인지 간단히 점검
본격적인 이상치 분석은 `02_eda.ipynb`에서 하지만, 단위를 잘못 이해하고 다음 단계로 넘어가면 전부 다시 해야 하니 여기서 최소한만 확인합니다: 기온이 켈빈(K) 단위인지(섭씨였다면 값이 마이너스나 30 근처여야 하는데, 켈빈이면 250~300대), 상대습도가 0~100 사이인지.

In [19]:
sample_temp_col_gfs = "gfs_g1_heightAboveGround_2_2t"
sample_temp_col_ldaps = "ldaps_g1_heightAboveGround_2_t"
sample_rh_col_gfs = "gfs_g1_heightAboveGround_2_2r"

print("GFS 기온(격자1) 범위:", train_base_wide[sample_temp_col_gfs].min(), "~", train_base_wide[sample_temp_col_gfs].max())
print("LDAPS 기온(격자1) 범위:", train_base_wide[sample_temp_col_ldaps].min(), "~", train_base_wide[sample_temp_col_ldaps].max())
print("GFS 상대습도(격자1) 범위:", train_base_wide[sample_rh_col_gfs].min(), "~", train_base_wide[sample_rh_col_gfs].max())

GFS 기온(격자1) 범위: 251.20033 ~ 305.82446
LDAPS 기온(격자1) 범위: 249.74907 ~ 302.03574
GFS 상대습도(격자1) 범위: 11.4 ~ 99.9


## 10. train/test 컬럼 구성이 일치하는지 검증
전처리 스킬의 핵심 규칙이에요: **라벨 컬럼을 제외하면 train과 test의 컬럼이 완전히 같아야** 합니다. 다르면 나중에 학습된 모델을 test에 그대로 못 씁니다 (컬럼이 안 맞아서 오류가 나거나, 더 나쁘게는 조용히 잘못된 값이 들어갑니다).

In [20]:
label_cols = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
common_id_cols = {"forecast_kst_dtm"}

train_cols_wide = set(train_base_wide.columns) - set(label_cols) - common_id_cols
test_cols_wide = set(test_base_wide.columns) - {"forecast_id"} - common_id_cols

only_in_train = train_cols_wide - test_cols_wide
only_in_test = test_cols_wide - train_cols_wide

print("train에만 있는 컬럼:", only_in_train)
print("test에만 있는 컬럼:", only_in_test)
assert only_in_train == set() and only_in_test == set(), "wide 버전 train/test 컬럼 구성이 다릅니다."
print("wide 버전 train/test 컬럼 구성 일치 확인 완료.")

train에만 있는 컬럼: set()
test에만 있는 컬럼: set()
wide 버전 train/test 컬럼 구성 일치 확인 완료.


In [21]:
train_cols_agg = set(train_base_agg.columns) - set(label_cols) - common_id_cols
test_cols_agg = set(test_base_agg.columns) - {"forecast_id"} - common_id_cols

only_in_train_agg = train_cols_agg - test_cols_agg
only_in_test_agg = test_cols_agg - train_cols_agg

assert only_in_train_agg == set() and only_in_test_agg == set(), "agg 버전 train/test 컬럼 구성이 다릅니다."
print("agg 버전 train/test 컬럼 구성 일치 확인 완료.")

agg 버전 train/test 컬럼 구성 일치 확인 완료.


## 11. parquet 캐시로 저장
여기까지 만든 표를 `data/processed/`에 저장합니다. parquet은 CSV보다 훨씬 빠르고 용량이 작으면서 컬럼 타입(dtype)을 그대로 보존하는 파일 형식이에요 (엑셀로 저장한 CSV처럼 타입이 깨질 걱정이 없습니다). 이 캐시들은 git에 올리지 않고(`.gitignore`), 각자의 컴퓨터에서 이 노트북을 다시 실행해서 만듭니다.

In [22]:
train_base_wide.to_parquet(PROCESSED_DIR / "train_base_wide.parquet", index=False)
train_base_agg.to_parquet(PROCESSED_DIR / "train_base_agg.parquet", index=False)
test_base_wide.to_parquet(PROCESSED_DIR / "test_base_wide.parquet", index=False)
test_base_agg.to_parquet(PROCESSED_DIR / "test_base_agg.parquet", index=False)

gfs_grid_meta.to_parquet(PROCESSED_DIR / "gfs_grid_meta.parquet", index=False)
ldaps_grid_meta.to_parquet(PROCESSED_DIR / "ldaps_grid_meta.parquet", index=False)

print("저장 완료:", sorted(p.name for p in PROCESSED_DIR.glob("*.parquet")))

저장 완료: ['gfs_grid_meta.parquet', 'ldaps_grid_meta.parquet', 'test_base_agg.parquet', 'test_base_wide.parquet', 'train_base_agg.parquet', 'train_base_wide.parquet']


In [23]:
_reload_check = pd.read_parquet(PROCESSED_DIR / "train_base_wide.parquet")
print("재로드 shape:", _reload_check.shape)
print("재로드 dtypes 샘플:")
print(_reload_check.dtypes.head())
del _reload_check

재로드 shape: (26304, 801)
재로드 dtypes 샘플:
forecast_kst_dtm                   datetime64[us]
kpx_group_1                               float64
kpx_group_2                               float64
kpx_group_3                               float64
gfs_g1_heightAboveGround_10_10u           float64
dtype: object


## 요약 — 이 노트북이 결정한 것 / 다음 노트북으로 넘기는 것

**결정한 것**
- GFS(9격자)/LDAPS(16격자) long 구조를 wide(격자별 컬럼 보존)와 agg(격자 평균·표준편차) 두 버전으로 만들어 둠 — 어느 쪽이 나은지는 아직 결정 안 함 (모델 비교로 결정)
- 병합 기준축은 항상 라벨(train)/제출 골격(test) — 기상 데이터 결측이 라벨 손실로 이어지지 않게 함
- 학습/검증 분리 날짜를 상수로 고정 (`TRAIN_START/END`, `VALID_START/END`) — 이후 모든 노트북이 재사용
- 결측치·이상치 처리, 스케일링은 아직 하지 않음 (다음 단계 근거 필요)

**다음 노트북(`02_eda.ipynb`)에 넘기는 것**
- 병합 후 결측이 어느 컬럼·기간에 몰려 있는지 (8번 셀 결과)
- 물리적 범위 점검 결과 (9번 셀 결과) — 단위가 예상과 다르면 EDA에서 더 깊게 확인 필요
- wide vs agg 버전 선택은 `eda-checklist`의 "풍속 계산 후 분포" 항목과 `model-selection` 단계에서 비교

**이해 체크**: "wide 버전과 agg 버전이 왜 둘 다 필요한지"를 한 문장으로 말할 수 있으면 이 노트북을 이해하신 거예요.